# 09 — Bodhan API Health Check (probe after the 'translate timeout' incident)

**Date:** 2026-11-19  
**Purpose:** verify whether `https://api.bodhan.ai` is reachable and all four paid model routes are alive, after the demo-run `ConnectTimeout` incident (LLM drafted an English summary → `translate_text(target_language='ta')` → `api.bodhan.ai` unreachable after ~23.6s).

**Cost cap:** ≤ ₹0.50. This notebook makes the smallest possible authenticated calls: 1 translate, 1 transcription, 1 synthesis, 1 OCR.  

**Outputs:** writes `experiments/outputs/exp09_api_health_check/health_report.json`.

**Free checks first, paid checks second.** The paid cells can be skipped by setting `RUN_PAID = False`.


In [ ]:
import socket, time, os, json, pathlib, hashlib
from datetime import datetime, timezone

try:
    import httpx
except ImportError:
    raise SystemExit('httpx missing; run with the recipe venv (.venv).')

RECIPE = pathlib.Path(r'D:\01_WORK\Career\Assessments\BodhanAI\ref\Bodhan-AI-Cookbook\examples\bodhan_tool_agent')
OUT = RECIPE / 'experiments' / 'outputs' / 'exp09_api_health_check'
OUT.mkdir(parents=True, exist_ok=True)

# --- Config (values never printed) ---------------------------------------
env = {}
for line in (RECIPE / '.env').read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if not line or line.startswith('#') or '=' not in line:
        continue
    k, v = line.split('=', 1)
    env[k.strip()] = v.strip().strip('"').strip("'")

KEYS = ['BODHAN_TRANSCRIBE_KEY', 'BODHAN_TRANSLATE_KEY', 'BODHAN_SPEAK_KEY', 'BODHAN_OCR_KEY']
print('Keys present:', {k: bool(env.get(k)) for k in KEYS})
print('BODHAN_BASE_URL:', env.get('BODHAN_BASE_URL'))
BASE = env.get('BODHAN_BASE_URL', '').rstrip('/')
assert BASE, 'BODHAN_BASE_URL missing in .env'
RUN_PAID = True
results = {'generated_at': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'), 'base_url': BASE, 'probes': [], 'paid_calls': []}

def record_probe(section, name, ok, detail, latency_ms=None):
    results[section].append({'name': name, 'ok': ok, 'detail': str(detail)[:300], 'latency_ms': latency_ms})
    print(f'  [{ "PASS" if ok else "FAIL" }] {name}: {str(detail)[:140]} ({latency_ms} ms)' if latency_ms else f'  [{ "PASS" if ok else "FAIL" }] {name}: {str(detail)[:140]}')


## 1. Free checks — DNS, TCP, and unauthenticated route probes

Unauthenticated calls must answer **401** (route alive) or, for the speculative `/translate` fallback, **404** (path does not exist on production). Anything else means the gateway is not answering.


In [ ]:
print('Free probes (no key, zero credit spend)')

# 1a. DNS
t0 = time.perf_counter()
try:
    addrs = sorted({i[4][0] for i in socket.getaddrinfo('api.bodhan.ai', 443, proto=socket.IPPROTO_TCP)})
    record_probe('probes', 'dns_resolve', bool(addrs), ','.join(addrs), int((time.perf_counter() - t0) * 1000))
except Exception as e:
    record_probe('probes', 'dns_resolve', False, e)

# 1b. TCP + TLS handshake
t0 = time.perf_counter()
try:
    with socket.create_connection(('api.bodhan.ai', 443), timeout=10):
        record_probe('probes', 'tcp_connect_443', True, 'OK', int((time.perf_counter() - t0) * 1000))
except Exception as e:
    record_probe('probes', 'tcp_connect_443', False, e)

# 1c. HTTPS GET /
t0 = time.perf_counter()
try:
    r = httpx.get('https://api.bodhan.ai/', timeout=10.0)
    record_probe('probes', 'GET_root', r.status_code == 200, f'HTTP {r.status_code}', int((time.perf_counter() - t0) * 1000))
except Exception as e:
    record_probe('probes', 'GET_root', False, e)

# 1d. Routes without auth — expect 401 (or 404 for the dead legacy path)
unauth = [
    ('POST_transcriptions', BASE + '/audio/transcriptions'),
    ('POST_chat_completions', BASE + '/chat/completions'),
    ('POST_speech', BASE + '/audio/speech'),
    ('POST_translate_root_legacy', BASE.rsplit('/v1', 1)[0] + '/translate'),
]
for name, url in unauth:
    t0 = time.perf_counter()
    try:
        r = httpx.post(url, json={}, timeout=10.0)
        body = r.text[:200]
        record_probe('probes', name, r.status_code in (400, 401, 404, 405), f'HTTP {r.status_code}: {body}', int((time.perf_counter() - t0) * 1000))
    except Exception as e:
        record_probe('probes', name, False, e)


## 2. Authenticated minimal calls (≤ ₹0.20 of spend)

One call per model, smallest legal payloads. Costs: transcribe ≤ ₹0.10, translate ~₹0.0002, synthesize ~₹0.0000, OCR = ₹0.20. **Total worst case ≈ ₹0.30.**


In [ ]:
if RUN_PAID:
    headers_t = {'Authorization': f"Bearer {env['BODHAN_TRANSLATE_KEY']}", 'Content-Type': 'application/json'}
    headers_s = {'Authorization': f"Bearer {env['BODHAN_SPEAK_KEY']}", 'Content-Type': 'application/json'}
    headers_o = {'Authorization': f"Bearer {env['BODHAN_OCR_KEY']}", 'Content-Type': 'application/json'}
    headers_r = {'Authorization': f"Bearer {env['BODHAN_TRANSCRIBE_KEY']}"}

    # 2a. translate: en -> ta via indic-translate, 1 sentence
    t0 = time.perf_counter()
    try:
        r = httpx.post(BASE + '/chat/completions', headers=headers_t, json={
            'model': 'indic-translate',
            'messages': [{'role': 'user', 'content': 'Translate the following text into Tamil. Reply with ONLY the Tamil translation, nothing else:\n\nThe office will be closed on Monday for maintenance.'}],
        }, timeout=60.0)
        ok = r.status_code == 200
        detail = r.text[:200]
        if ok:
            try:
                detail = r.json()['choices'][0]['message']['content'][:120]
            except Exception:
                pass
        record_probe('paid_calls', 'translate_en_ta', ok, f'HTTP {r.status_code}: {detail}', int((time.perf_counter() - t0) * 1000))
    except Exception as e:
        record_probe('paid_calls', 'translate_en_ta', False, e)
    print('  est. cost ≈ ₹0.001')

    # 2b. transcribe: gujarati short burst
    t0 = time.perf_counter()
    try:
        clip = RECIPE / 'sample_data' / 'speech' / 'sample_speech_06_gu_short_burst.wav'
        assert clip.exists(), f'missing clip {clip}'
        with clip.open('rb') as f:
            r = httpx.post(BASE + '/audio/transcriptions', headers=headers_r,
                           files={'file': ('clip.wav', f, 'audio/wav')},
                           data={'model': 'indic-transcribe'}, timeout=60.0)
        ok = r.status_code == 200
        detail = r.text[:200]
        if ok:
            try:
                detail = (r.json().get('text') or str(r.json()))[:120]
            except Exception:
                pass
        record_probe('paid_calls', 'transcribe_gu_clip', ok, f'HTTP {r.status_code}: {detail}', int((time.perf_counter() - t0) * 1000))
    except Exception as e:
        record_probe('paid_calls', 'transcribe_gu_clip', False, e)
    print('  est. cost ≤ ₹0.10')

    # 2c. synthesize: one short English sentence, voice Kavya (instructions are de-facto required)
    t0 = time.perf_counter()
    try:
        r = httpx.post(BASE + '/audio/speech', headers=headers_s, json={
            'model': 'indic-speak',
            'input': 'The office will be closed on Monday.',
            'voice': 'Kavya',
            'instructions': json.dumps({'lang': 'en', 'style': 'news'}),
        }, timeout=60.0)
        ok = r.status_code == 200
        detail = f'HTTP {r.status_code}, {len(r.content)} bytes audio'
        record_probe('paid_calls', 'synthesize_en', ok, detail, int((time.perf_counter() - t0) * 1000))
    except Exception as e:
        record_probe('paid_calls', 'synthesize_en', False, e)
    print('  est. cost ≈ ₹0.0000')

    # 2d. OCR: one Hindi invoice scan (₹0.20)
    t0 = time.perf_counter()
    try:
        import base64
        img = RECIPE / 'sample_data' / 'images' / 'sample_img_01_hi_invoice.png'
        assert img.exists(), f'missing image {img}'
        b64 = base64.b64encode(img.read_bytes()).decode()
        r = httpx.post(BASE + '/chat/completions', headers=headers_o, json={
            'model': 'indic-ocr',
            'messages': [{'role': 'user', 'content': [
                {'type': 'text', 'text': 'Extract all text from this image.'},
                {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{b64}'}},
            ]}],
        }, timeout=120.0)
        ok = r.status_code == 200
        detail = r.text[:200]
        if ok:
            try:
                detail = r.json()['choices'][0]['message']['content'][:150]
            except Exception:
                pass
        record_probe('paid_calls', 'ocr_hi_invoice', ok, f'HTTP {r.status_code}: {detail}', int((time.perf_counter() - t0) * 1000))
    except Exception as e:
        record_probe('paid_calls', 'ocr_hi_invoice', False, e)
    print('  est. cost = ₹0.20')
else:
    print('RUN_PAID=False — skipped. Only free probes were run.')


## 3. Verdict

Writes `health_report.json` and prints the single-sentence conclusion for the PR/debrief.


In [ ]:
results['summary'] = {
    'dns_ok': next((p for p in results['probes'] if p['name'] == 'dns_resolve'), {}).get('ok'),
    'tcp_ok': next((p for p in results['probes'] if p['name'] == 'tcp_connect_443'), {}).get('ok'),
    'get_root_ok': next((p for p in results['probes'] if p['name'] == 'GET_root'), {}).get('ok'),
    'routes_answered': [p['name'] for p in results['probes'] if p['name'].startswith('POST_') and p['ok']],
    'paid_calls_ok': [p['name'] for p in results['paid_calls'] if p['ok']],
    'paid_calls_failed': [p['name'] for p in results['paid_calls'] if not p['ok']],
}
report_path = OUT / 'health_report.json'
report_path.write_text(json.dumps(results, indent=2) + chr(10), encoding='utf-8')
print(json.dumps(results['summary'], indent=2))
print()
print('Report written to:', report_path)

if results['paid_calls']:
    n_ok = sum(1 for p in results['paid_calls'] if p['ok'])
    print(f'Paid calls: {n_ok}/{len(results["paid_calls"])} succeeded')
    if n_ok == len(results['paid_calls']):
        print('VERDICT: Bodhan API is healthy right now. The incident was transient or client-side; record it with timestamps, keep Track D retry behavior, and do not re-architecture around an outage that no longer reproduces.')
    else:
        print('VERDICT: partial availability — see failed calls above. That narrows the outage surface; do not re-run paid cells blindly.')


## 4. Notes (findings from the 2026-11-19 run)

- **No Bodhan outage at the time of probing.** translate/transcribe/synthesize/OCR all answered authenticated calls within ~1.3s; DNS/TCP fine.
- **Indic-Speak requires `instructions`** (a JSON string, per the API spec). Sending none → **HTTP 500** with a generic `{"error":{"message":"Internal server error"}}`. The 2026-11-19 500 was this, not an outage.
- **Style drift:** the spec lists `formal` among styles, but `instructions={"lang":"en","style":"formal"}` returns **HTTP 500** on the live service (English input). Treat the listed style catalog as *indicative only*; prefer documented working styles (e.g., `news`).
- The `/translate` (root) fallback 404s on production — the direct-translation path does not exist; `indic-translate` speaks only via `/v1/chat/completions` here.
- `ConnectTimeout` with `httpx.Timeout(connect=5.0)` is consistent with `~5s × attempts ≈ 23.6s`. That is why the resilience track treats `ConnectTimeout` as retryable-but-bounded rather than proof of a provider outage.
- `transcribe_gu_clip` returned a `হুম হুম…` loop because `sample_speech_06_gu_short_burst.wav` carries no speech (synthetic tone); the model looped rather than error — a data artifact, not an API failure.
- Re-run this notebook any time with:

```powershell
.venv\Scripts\jupyter.exe nbconvert --to notebook --execute --inplace experiments/09_bodhan_api_health_check.ipynb
```
